In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="mistral",  
)

In [5]:
from langchain_community.document_loaders import WebBaseLoader

web_loader = WebBaseLoader("https://www.geeksforgeeks.org/machine-learning/machine-learning/")

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
data = web_loader.load()


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)

In [8]:
data_split = text_splitter.split_documents(data)

data_split

[Document(metadata={'source': 'https://www.geeksforgeeks.org/machine-learning/machine-learning/', 'title': 'Machine Learning Tutorial - GeeksforGeeks', 'description': 'Your All-in-One Learning Portal: GeeksforGeeks is a comprehensive educational platform that empowers learners across domains-spanning computer science and programming, school education, upskilling, commerce, software tools, competitive exams, and more.', 'language': 'en'}, page_content='Machine Learning Tutorial - GeeksforGeeksCoursesTutorialsPracticeJobsPython for Machine LearningMachine Learning with RMachine Learning AlgorithmsEDAMath for Machine LearningMachine Learning Interview QuestionsML ProjectsDeep LearningNLPComputer visionData ScienceArtificial IntelligenceShare Your ExperiencesMachine Learning BasicsIntroductionTypesML PipelineApplicationsPython for Machine LearningML with PythonNumpyPandasData PreprocessingEDAFeature EngineeringFeature EngineeringDimensionality ReductionFeature SelectionSupervised LearningS

In [12]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
embedding = OllamaEmbeddings(model = 'embeddinggemma:latest')

vector_db = FAISS.from_documents(data_split, embedding)

In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(
    """
You are an AI Assistant, your job is to Based on the contect data provide me the response for the input 
<CONTEXT>
{context}
</CONTEXT>
    
    """
)

document_chain = create_stuff_documents_chain(llm, prompt_template)

In [15]:
from langchain_classic.chains import create_retrieval_chain

vector_db_retrieval = vector_db.as_retriever()

chain = create_retrieval_chain()

In [9]:
from langchain_core.prompts import ChatPromptTemplate

# Extract just the page content from documents to avoid metadata template variables
data_content = "\n".join([doc.page_content for doc in data])

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a agent that will answer my query based on the data given. If you cannot find the answer or any irrelavent queries then answer it as 'There is no data for this query, please ask a different query.'"),
        ("system", "data : " + data_content),
        ("user", "{prompt}")
    ]
)

chain = prompt_template | llm

In [14]:
query = "what is Learning?"
result = chain.invoke({"prompt" : query})
